<a href="https://colab.research.google.com/github/jclevitt1/neoantigen-pipeline/blob/stage-2a-and-interactive-viz/notebooks/option_B_integration_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Option B — full-pipeline integration test on a real tumour (HCC1395)

Where **Option A** hand-fed curated mutations into stages 3→6, Option B runs the
**genomics front end on a real tumour** (the SEQC2 HCC1395 tumor/normal pair) and
then the same validated back half. Two clearly separated halves:

**Part 1 — STAGE-2 TEST (front end).** acquire a chr21 slice of the real BAMs →
sort/index → **Mutect2** call → **VEP + Wildtype** annotate → a real annotated
somatic VCF. Closed by a single programmatic gate (`assert_stage2_vcf`).

**Part 2 — VACCINE CONSTRUCTION.** feed that real VCF into stages 3→6 (the
Option-A-validated path) → ranked neoepitopes + string-of-beads construct.

**Honest scope.** DNA-only (no RNA slice) → expression runs *permissive*; chr21-only
→ a real **slice** of the tumour, not the whole genome. Both stated in the output.

> ⚠️ **Part 1 setup is the real work/risk** — VEP + the ~15 GB GRCh38 cache + the
> Wildtype plugin, GATK, and a contig-matched reference. Expect 2–3 iterations.
> Part 2 is known-good. The design intent (see `docs/e2e_validation_notes.md`) is that
> the heavy tools are **transcribed** here, not run through the pipeline object.

In [ ]:
# --- Get the PRIVATE repo onto Colab and on sys.path (it has core.py at its root) ---
# Clones if missing (prompts for a GitHub token with repo scope; the URL is never
# printed, so the token doesn't leak), else pulls latest. A fresh Colab runtime wipes
# /content, so this must run before any `import NeoantigenVaccineConstructionPipeline`.
import os, sys, importlib, subprocess, getpass
REPO = '/content/neoantigen_pipeline'
BRANCH = 'stage-2a-and-interactive-viz'
if not os.path.isdir(REPO):
    tok = os.environ.get('GH_TOKEN') or getpass.getpass('GitHub token: ')
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
        f'https://{tok}@github.com/jclevitt1/neoantigen-pipeline.git', REPO],
        capture_output=True, text=True)
    print('clone OK' if r.returncode == 0 else 'clone FAILED — check token/branch')
else:
    subprocess.run(['git', '-C', REPO, 'pull'], capture_output=True, text=True)
    print('pulled latest')
if REPO not in sys.path:
    sys.path.insert(0, REPO)
importlib.invalidate_caches()   # CRITICAL when files/dirs were ADDED (bit us on Option A)

def refresh(repo=REPO, pkgs=('NeoantigenVaccineConstructionPipeline', 'core')):
    """Pull latest + evict cached modules so new files/dirs are seen. Re-import after."""
    print(subprocess.run(['git', '-C', repo, 'pull'], capture_output=True, text=True).stdout)
    for name in list(sys.modules):
        if any(name == p or name.startswith(p + '.') for p in pkgs):
            del sys.modules[name]
    importlib.invalidate_caches()
    print('refreshed — now RE-IMPORT the names you use')

import NeoantigenVaccineConstructionPipeline as _p
print('import OK ->', _p.__file__)

---
## PART 1 — STAGE-2 TEST (the genomics front end)

Everything from here to the **stage-2 gate** exercises stage 2 (acquire → call →
annotate). If the gate passes, the front end is proven and we move to construction.

In [ ]:
# 1. samtools, and CONFIRM CONTIG NAMING before anything else (chr21 vs 21).
#    The BAM, the reference, and the VEP cache must all agree — classic footgun.
!apt-get -qq install -y samtools > /dev/null
import re, subprocess
from NeoantigenVaccineConstructionPipeline.stages.acquire.seqc2_slice.source import (
    seqc2_wes_dna_manifest)
TUMOR_URL = seqc2_wes_dna_manifest(center='EA')['tumor_dna']
hdr = subprocess.run(['samtools', 'view', '-H', TUMOR_URL], capture_output=True, text=True).stdout
sq = [l for l in hdr.splitlines() if l.startswith('@SQ')][:3]
print('\n'.join(sq))
REGION = 'chr21' if any('SN:chr' in l for l in hdr.splitlines() if l.startswith('@SQ')) else '21'
print('\n-> using REGION =', REGION)

In [ ]:
# 2. GATK4 (Java jar; the standard Colab install is the release zip).
GATK_VER = '4.5.0.0'
!wget -q https://github.com/broadinstitute/gatk/releases/download/{GATK_VER}/gatk-{GATK_VER}.zip -O /content/gatk.zip
!unzip -q -o /content/gatk.zip -d /content
GATK = f'/content/gatk-{GATK_VER}/gatk'
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!{GATK} --version

In [ ]:
# 3. Ensembl VEP + the Wildtype plugin + the GRCh38 cache. ⚠️ THE CRUX / SLOW / ~15 GB.
#    VEP is Perl, so bioconda is the least-pain install in Colab. Expect this to be the
#    cell you iterate on. The Wildtype plugin is what puts WildtypeProtein in the CSQ.
!pip -q install condacolab
import condacolab; condacolab.install()   # NB: restarts the kernel; re-run from cell 2 after.


In [ ]:
# 3b. (after the condacolab kernel restart + re-running cell 2) install VEP, plugin, cache.
!mamba install -y -q -c bioconda -c conda-forge ensembl-vep
VEP_CACHE = '/content/vep_cache'; VEP_PLUGINS = '/content/vep_plugins'
# cache (offline annotation data) + the Wildtype plugin:
!vep_install -a cf -s homo_sapiens -y GRCh38 -c {VEP_CACHE} --NO_HTSLIB -n
!vep_install -a p --PLUGINS Wildtype -g Wildtype --PLUGINSDIR {VEP_PLUGINS} -n
print('VEP cache:', VEP_CACHE, '| plugins:', VEP_PLUGINS)

In [ ]:
# 4. A GRCh38 reference for Mutect2, with .fai + .dict. It must be CONTIG-COMPATIBLE
#    with the SEQC2 BAM header (see REGION check above). The Broad hg38 analysis set is
#    the safe match for SEQC2 best-practices BAMs (chr-prefixed). ~3 GB.
!mkdir -p /content/ref
REF = '/content/ref/Homo_sapiens_assembly38.fasta'
BROAD = 'https://storage.googleapis.com/genomics-public-data/resources/broad/hg38/v0'
!wget -q {BROAD}/Homo_sapiens_assembly38.fasta       -O {REF}
!wget -q {BROAD}/Homo_sapiens_assembly38.fasta.fai   -O {REF}.fai
!wget -q {BROAD}/Homo_sapiens_assembly38.dict        -O /content/ref/Homo_sapiens_assembly38.dict
print('reference ready:', REF)
# If GATK later complains about a sequence-dictionary mismatch, the BAM was aligned to a
# different GRCh38 build — use that exact reference instead.

In [ ]:
# 5. Acquire: samtools region-slice the real SEQC2 tumor+normal BAMs (no full download).
from NeoantigenVaccineConstructionPipeline.demos import integration_test as B
RUN = '/content/run_B'
case = B.make_case(RUN, proteome='/content/human.fasta', reference=REF)
B.build_acquire(center='EA', region=REGION).ensure(case)   # -> case.tumor_dna / normal_dna
!ls -lh {case.tumor_dna} {case.normal_dna}

In [ ]:
# 6. Stage-1 normalize (pre-aligned BAM → sort+index only; transcribed plan). Also grab
#    each BAM's read-group sample name (SM) — Mutect2 needs the NORMAL sample id.
for raw, bam in [(case.tumor_dna, case.tumor_dna_bam), (case.normal_dna, case.normal_dna_bam)]:
    !samtools sort -o {bam} {raw} && samtools index {bam}
def sm(bam):
    h = subprocess.run(['samtools','view','-H',str(bam)], capture_output=True, text=True).stdout
    m = re.search(r'SM:(\S+)', h); return m.group(1) if m else None
TUMOR_SM, NORMAL_SM = sm(case.tumor_dna_bam), sm(case.normal_dna_bam)
print('tumor SM =', TUMOR_SM, '| normal SM =', NORMAL_SM)

In [ ]:
# 7. Stage-2a: call (Mutect2, matched normal, restricted to REGION) → filter → annotate
#    (VEP + Wildtype). This is the transcription of Mutect2VepCaller.COMMAND_PLAN,
#    updated to GATK4 syntax (--normal-sample <SM>; -tumor is inferred).
RAW = f'{RUN}/mutect2.raw.vcf.gz'
FILT = f'{RUN}/mutect2.filtered.vcf.gz'
OUT = str(case.somatic_vcf)   # stage 3 reads exactly this path
!{GATK} Mutect2 -R {REF} -I {case.tumor_dna_bam} -I {case.normal_dna_bam} \
    --normal-sample {NORMAL_SM} -L {REGION} -O {RAW}
!{GATK} FilterMutectCalls -R {REF} -V {RAW} -O {FILT}
!vep -i {FILT} -o {OUT} --vcf --force_overwrite \
    --symbol --transcript_version --hgvs \
    --plugin Wildtype --dir_plugins {VEP_PLUGINS} \
    --fasta {REF} --cache --offline --assembly GRCh38 --dir_cache {VEP_CACHE}
!grep -v '^##' {OUT} | head

In [ ]:
# 8. ===== STAGE-2 GATE — tests end here =====
#    Proves the front end produced a USABLE annotated VCF (exists, non-empty, CSQ
#    declares WildtypeProtein, ≥1 record, ≥1 record actually carries a WT protein).
#    Raises with a specific reason if Part 1 is wrong, so we never feed junk downstream.
B.assert_stage2_vcf(case)

---
## PART 2 — VACCINE CONSTRUCTION (the validated back half)

Stages 3→6 over the real `case.somatic_vcf` from Part 1. Identical code path to
Option A — the only new thing is that the input mutations came from real reads.

In [ ]:
# 9. MHCflurry (presentation gate) + a real human proteome (Łuksza dissimilarity term).
!pip -q install mhcflurry
!mhcflurry-downloads fetch models_class1_presentation
!wget -qO /content/human.fasta.gz 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&compressed=true&query=organism_id:9606+AND+reviewed:true'
!gunzip -f /content/human.fasta.gz
!grep -c '^>' /content/human.fasta  # sanity: ~20k proteins

In [ ]:
# 10. Run stages 2b(known HLA) → 3 → 4 → 5 → 6 on the real VCF.
from NeoantigenVaccineConstructionPipeline.demos import integration_test as B
out = B.run_construction(RUN, proteome='/content/human.fasta', reference=REF)
out

In [ ]:
# 11. Inspect: the ranked neoepitopes from a REAL tumour, and the vaccine construct.
import pandas as pd
print('=== RANKED (stage 4) ===')
display(pd.read_csv(out['ranked_tsv'], sep='\t').head(20))
print('=== FILTERED (stage 5 survivors) ===')
display(pd.read_csv(out['filtered_tsv'], sep='\t'))
print('=== CONSTRUCT (stage 6) ===')
print(out['construct_fasta'].read_text())